In [ ]:
import pandas as pd

In [ ]:
flights = pd.read_parquet(
    "../data/processed/flights.parquet"
)

weather = pd.read_parquet(
    "../data/processed/weather.parquet"
)

In [ ]:
flights[
    ["origin_airport", "scheduled_departure"]
].dtypes

In [ ]:
weather[
    ["airport_icao", "datetime", "timezone"]
].dtypes

In [ ]:
print(f"Flights: {flights.shape}")
print(f"Weather: {weather.shape}")

In [ ]:
airport_timezones = (
    weather[
        ["airport_icao", "timezone"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

airport_timezones.head()

In [ ]:
airport_timezones.shape

In [ ]:
airport_timezones[
    "airport_icao"
].value_counts().head()

In [ ]:
airport_timezones["timezone"].value_counts()

In [ ]:
flights = flights.merge(
    airport_timezones,
    left_on="origin_airport",
    right_on="airport_icao",
    how="left",
    validate="many_to_one",
)

In [ ]:
print(flights.shape)

flights["timezone"].isna().sum()

In [ ]:
flights["scheduled_departure_local"] = pd.NaT

for timezone, index in flights.groupby("timezone").groups.items():
    local_time = (
        flights.loc[index, "scheduled_departure"]
        .dt.tz_localize("America/Sao_Paulo")
        .dt.tz_convert(timezone)
        .dt.tz_localize(None)
    )

    flights.loc[index, "scheduled_departure_local"] = local_time

In [ ]:
flights["scheduled_departure_hour"] = (
    flights["scheduled_departure_local"]
    .dt.floor("h")
)

In [ ]:
for airport in ["SBGR", "SBEG", "SBRB"]:
    display(
        flights.loc[
            flights["origin_airport"] == airport,
            [
                "origin_airport",
                "scheduled_departure",
                "timezone",
                "scheduled_departure_local",
                "scheduled_departure_hour",
            ],
        ].head(5)
    )

In [ ]:
weather.duplicated(
    subset=["airport_icao", "datetime"]
).sum()

In [ ]:
flight_weather = flights.merge(
    weather,
    left_on=[
        "origin_airport",
        "scheduled_departure_hour",
    ],
    right_on=[
        "airport_icao",
        "datetime",
    ],
    how="left",
    validate="many_to_one",
)

In [ ]:
print(f"Flights antes: {len(flights):,}")
print(f"Após merge: {len(flight_weather):,}")

In [ ]:
flight_weather["precipitation"].isna().sum()

In [ ]:
flight_weather["precipitation"].isna().mean()

In [ ]:
flight_weather.loc[
    flight_weather["precipitation"].isna(),
    [
        "origin_airport",
        "scheduled_departure",
        "scheduled_departure_local",
        "scheduled_departure_hour",
    ],
].head(20)

In [ ]:
missing_weather = flight_weather[
    flight_weather["precipitation"].isna()
]

missing_weather["scheduled_departure"].isna().value_counts()

In [ ]:
pd.crosstab(
    flight_weather["scheduled_departure"].isna(),
    flight_weather["precipitation"].isna(),
)

In [ ]:
[col for col in flight_weather.columns if "timezone" in col]

In [ ]:
missing_with_departure = flight_weather[
    flight_weather["precipitation"].isna()
    & flight_weather["scheduled_departure"].notna()
]

missing_with_departure[
    [
        "origin_airport",
        "scheduled_departure",
        "timezone_x",
        "scheduled_departure_local",
        "scheduled_departure_hour",
    ]
].sort_values("scheduled_departure")

In [ ]:
(flight_weather["timezone_x"] == flight_weather["timezone_y"]).value_counts(dropna=False)

In [ ]:
flights["scheduled_departure"].agg(["min", "max"])

In [ ]:
flights[
    flights["scheduled_departure"].dt.year > 2025
]["scheduled_departure"].dt.year.value_counts()

In [ ]:
flight_weather = flight_weather[
    flight_weather["scheduled_departure"].between(
        "2022-01-01",
        "2025-12-31 23:59:59"
    )
].copy()

In [ ]:
flight_weather = flight_weather[
    flight_weather["scheduled_departure"].notna()
].copy()

In [ ]:
flight_weather.shape

In [ ]:
flight_weather["precipitation"].isna().sum()

In [ ]:
(flight_weather["timezone_x"]== flight_weather["timezone_y"]).value_counts(dropna=False)

In [ ]:
flight_weather = flight_weather.drop(
    columns=[
        "airport_icao_y",
        "timezone_y",
    ],
    errors="ignore",
)

flight_weather = flight_weather.rename(
    columns={
        "airport_icao_x": "airport_icao",
        "timezone_x": "timezone",
    }
)

In [ ]:
flight_weather.columns.tolist()

In [ ]:
flight_weather.shape

In [ ]:
flight_weather.columns[flight_weather.columns.duplicated()].tolist()

In [ ]:
flight_weather = flight_weather.loc[
    :, ~flight_weather.columns.duplicated()
].copy()

In [ ]:
flight_weather.columns.is_unique

In [ ]:
flight_weather.shape

In [ ]:
flight_weather.to_parquet(
    "../data/processed/flight_weather.parquet",
    index=False,
)